# 04 — Merge and Align EU and US Panels

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 1486619\*  

**Purpose:** Load the intermediate outputs from notebooks 03a (EU) and 03b (US), validate their schemas, construct the one-day lag structure required for the spillover regression, and save a merged analysis-ready dataset.

**Inputs:**
- `data/intermediate/eu2013_analysis_ready__<timestamp>.csv`
- `data/intermediate/us2014_analysis_ready__<timestamp>.csv`

**Output:**
- `data/analysis_ready/panel_merged__<timestamp>.csv`
- `logs/04_merge_log__<timestamp>.json`

**Prerequisites:** Notebooks 03a and 03b must have been run and their outputs must exist in `data/intermediate/`.

---

## Lag structure rationale

European equity markets (e.g., Xetra/Frankfurt) close at approximately 17:30 CET, which corresponds to 11:30 EST. US equity markets (NYSE/NASDAQ) close at 16:00 EST. On the same calendar day $t$, the US market closes approximately 4.5 hours **after** the European market has already closed.

This creates a natural information ordering:
- EU smile parameters on day $t$ are determined **before** US markets close on day $t$.
- US smile parameters on day $t$ are determined **after** EU markets have closed on day $t$.

The spillover hypothesis is therefore directional: **US smile on day $t$ predicts EU smile on day $t+1$**.

The merge key is:
```
EU date (t+1) matched to US date (t)
```
Equivalently: US variables are lagged by one trading day before merging with EU variables.

**Important constraint:** The EU sample (2013) and the US sample (2014) do **not** overlap in calendar time. This notebook builds the correct merge infrastructure and validates it. A meaningful spillover regression requires overlapping samples — this will only be possible once full OptionMetrics access is obtained or an alternative data source is used. The current output will be a correctly structured but empty (zero matched rows) merged panel, with full schema preserved.

---
## Step 0 — Imports and configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import json
from datetime import datetime

# Timestamp for all outputs in this notebook
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Paths
INTERMEDIATE_DIR = '../data/intermediate/'
OUTPUT_DIR       = '../data/analysis_ready/'
LOG_DIR          = '../logs/'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Log dict — everything gets written here and saved at the end
log = {
    'notebook': '04_merge_align',
    'run_timestamp': RUN_TIMESTAMP,
    'steps': {}
}

print(f'Run timestamp: {RUN_TIMESTAMP}')

---
## Step 1 — Load intermediate files

Both files are loaded by picking the most recent timestamped version in `data/intermediate/`. If multiple versions exist (from multiple runs of 03a/03b), the most recent is used. This is logged explicitly.

In [ ]:
def load_latest(prefix, directory):
    """
    Load the most recently modified CSV file matching a given prefix.
    Returns (DataFrame, filepath).
    Raises FileNotFoundError if no matching file exists.
    """
    pattern = os.path.join(directory, f'{prefix}*.csv')
    matches = glob.glob(pattern)
    if len(matches) == 0:
        raise FileNotFoundError(
            f'No file matching {pattern} found. '
            f'Make sure notebook 03a/03b has been run first.'
        )
    # Sort by modification time, take most recent
    latest = max(matches, key=os.path.getmtime)
    df = pd.read_csv(latest, parse_dates=['date'])
    print(f'Loaded: {latest}')
    print(f'  Shape: {df.shape}')
    print(f'  Date range: {df["date"].min().date()} to {df["date"].max().date()}')
    print(f'  Trading days: {df["date"].nunique()}')
    print(f'  Maturity nodes: {sorted(df["days"].unique().tolist())}')
    print()
    return df, latest

# Load EU panel
print('--- EU Panel ---')
eu, eu_path = load_latest('eu2013_analysis_ready__', INTERMEDIATE_DIR)

# Load US panel
print('--- US Panel ---')
us, us_path = load_latest('us2014_analysis_ready__', INTERMEDIATE_DIR)

log['steps']['step1_load'] = {
    'eu_file': eu_path,
    'eu_shape': list(eu.shape),
    'eu_date_min': str(eu['date'].min().date()),
    'eu_date_max': str(eu['date'].max().date()),
    'us_file': us_path,
    'us_shape': list(us.shape),
    'us_date_min': str(us['date'].min().date()),
    'us_date_max': str(us['date'].max().date()),
}

---
## Step 2 — Schema validation

Before merging, verify that both panels have identical column structures and compatible dtypes. Any schema mismatch here is a pipeline error that must be fixed upstream in 03a or 03b — do not patch it here.

In [ ]:
# Expected columns — these must exist in both panels
# Based on the validated output of 03a (110 rows x 21 cols)
REQUIRED_COLS = [
    'date',
    'days',
    'atm_iv',
    'skew',
    'curvature',
    'rf_rate',
    'vix',
    'eurusd',
]

# Check EU
eu_missing = [c for c in REQUIRED_COLS if c not in eu.columns]
# Check US
us_missing = [c for c in REQUIRED_COLS if c not in us.columns]

if eu_missing:
    raise ValueError(f'EU panel missing required columns: {eu_missing}')
if us_missing:
    raise ValueError(f'US panel missing required columns: {us_missing}')

print('Required columns present in both panels. ✓')
print()
print('EU columns:', eu.columns.tolist())
print()
print('US columns:', us.columns.tolist())
print()

# Check dtypes for key columns
print('EU date dtype:', eu['date'].dtype)
print('US date dtype:', us['date'].dtype)
assert pd.api.types.is_datetime64_any_dtype(eu['date']), 'EU date column is not datetime'
assert pd.api.types.is_datetime64_any_dtype(us['date']), 'US date column is not datetime'
print('Date dtypes are datetime64. ✓')
print()

# Maturity nodes — must be identical between panels
eu_days = sorted(eu['days'].unique().tolist())
us_days = sorted(us['days'].unique().tolist())
print(f'EU maturity nodes: {eu_days}')
print(f'US maturity nodes: {us_days}')
if eu_days != us_days:
    print('WARNING: Maturity nodes differ between EU and US panels.')
    print('The merge will proceed but only matching nodes will appear in the output.')
    print('This is expected if OptionMetrics surfaces differ between samples.')
else:
    print('Maturity nodes are identical. ✓')

log['steps']['step2_schema'] = {
    'eu_columns': eu.columns.tolist(),
    'us_columns': us.columns.tolist(),
    'eu_maturity_nodes': eu_days,
    'us_maturity_nodes': us_days,
    'nodes_identical': eu_days == us_days,
}

---
## Step 3 — Check for date overlap

The spillover regression requires overlapping calendar dates. This step checks whether the two panels share any trading dates. With sample data (EU=2013, US=2014), there will be zero overlap. This is not an error in this notebook — it is a data access constraint documented here explicitly.

In [ ]:
eu_dates = set(eu['date'].dt.date.unique())
us_dates = set(us['date'].dt.date.unique())

overlapping_dates = eu_dates.intersection(us_dates)
n_overlap = len(overlapping_dates)

print(f'EU trading days:  {len(eu_dates)}')
print(f'US trading days:  {len(us_dates)}')
print(f'Overlapping days: {n_overlap}')
print()

if n_overlap == 0:
    print('NO OVERLAPPING DATES FOUND.')
    print()
    print('This is expected with current sample data:')
    print(f'  EU window: {min(eu_dates)} to {max(eu_dates)}')
    print(f'  US window: {min(us_dates)} to {max(us_dates)}')
    print()
    print('The merge infrastructure below is correct and will produce a non-empty')
    print('output once overlapping data is available (full OptionMetrics access).')
    print('The output file will be saved with zero data rows but correct schema.')
else:
    print(f'Overlap found: {sorted(overlapping_dates)}')

log['steps']['step3_overlap'] = {
    'n_eu_dates': len(eu_dates),
    'n_us_dates': len(us_dates),
    'n_overlapping_dates': n_overlap,
    'overlap_warning': n_overlap == 0,
}

---
## Step 4 — Construct the lag structure

**Logic:**
- For the EU panel: keep date as `date_eu` (this is the dependent variable date, i.e., day $t+1$).
- For the US panel: create a column `date_eu` = US date + 1 **trading day** (this is the date on which the US smile would predict the EU smile).
- Merge on `date_eu` and `days` (maturity node).

**Why trading days, not calendar days:**  
If the US observation falls on a Friday, the next trading day for EU is Monday (skipping weekend). We handle this by building an explicit trading calendar from the union of observed dates in both panels, then using `shift(1)` on that calendar.

**Prefix convention after merge:**
- EU columns: `eu_` prefix (dependent variables, day $t+1$)
- US columns: `us_` prefix (independent variables / regressors, day $t$)

In [ ]:
# Build trading calendar from union of observed dates in both panels
all_dates = sorted(eu_dates.union(us_dates))
trading_cal = pd.Series(all_dates, name='date')
trading_cal = pd.to_datetime(trading_cal)

print(f'Trading calendar: {len(trading_cal)} unique dates')
print(f'  From: {trading_cal.min().date()} to {trading_cal.max().date()}')
print()

# Build next-trading-day map
# For each date in the calendar, next_trading_day maps it to the following date
next_trading_day = dict(zip(trading_cal.iloc[:-1], trading_cal.iloc[1:]))

# Add lag column to US panel: date_eu = next trading day after US date
us_lagged = us.copy()
us_lagged['date_eu'] = us_lagged['date'].map(next_trading_day)

# Rows where the US date is the last date in the calendar have no next day
n_dropped_lag = us_lagged['date_eu'].isna().sum()
us_lagged = us_lagged.dropna(subset=['date_eu'])

print(f'US rows before lag: {len(us)}')
print(f'US rows dropped (last date, no next trading day): {n_dropped_lag}')
print(f'US rows after lag:  {len(us_lagged)}')
print()

# Rename date column in EU panel to date_eu for merge key
eu_keyed = eu.rename(columns={'date': 'date_eu'})

# Add prefixes before merge so columns don't collide
eu_cols = {c: f'eu_{c}' for c in eu_keyed.columns if c not in ['date_eu', 'days']}
us_cols = {c: f'us_{c}' for c in us_lagged.columns if c not in ['date_eu', 'days', 'date']}

eu_keyed = eu_keyed.rename(columns=eu_cols)
us_lagged = us_lagged.rename(columns=us_cols)

print('EU columns after prefix:', eu_keyed.columns.tolist())
print()
print('US columns after prefix:', us_lagged.columns.tolist())

log['steps']['step4_lag'] = {
    'trading_calendar_length': len(trading_cal),
    'us_rows_dropped_last_date': int(n_dropped_lag),
    'lag_convention': 'US date t -> EU date t+1 (next trading day)',
}

---
## Step 5 — Merge

Merge key: `['date_eu', 'days']`  
- `date_eu` is the EU observation date (= US date + 1 trading day)  
- `days` is the maturity node (must match exactly: 30, 60, 91, etc.)

Join type: `inner` — only matched (date_eu, days) pairs are kept.  
With non-overlapping samples, the result is zero rows. This is correct and expected.

In [ ]:
merged = pd.merge(
    eu_keyed,
    us_lagged,
    on=['date_eu', 'days'],
    how='inner'
)

print(f'Merged panel shape: {merged.shape}')
print(f'Rows: {len(merged)}')
print(f'Columns: {len(merged.columns)}')
print()

if len(merged) == 0:
    print('Zero rows in merged panel — expected with non-overlapping sample windows.')
    print('Schema is correct. Awaiting overlapping data.')
else:
    print(f'Date range in merged panel: {merged["date_eu"].min().date()} to {merged["date_eu"].max().date()}')
    print(f'Unique EU dates: {merged["date_eu"].nunique()}')
    print(f'Maturity nodes: {sorted(merged["days"].unique().tolist())}')
    print()
    print(merged[['date_eu', 'days', 'eu_atm_iv', 'eu_skew', 'eu_curvature',
                  'us_atm_iv', 'us_skew', 'us_curvature']].head(10))

log['steps']['step5_merge'] = {
    'merged_rows': len(merged),
    'merged_cols': len(merged.columns),
    'merge_key': ['date_eu', 'days'],
    'join_type': 'inner',
    'zero_rows_expected': len(merged) == 0,
}

---
## Step 6 — Validate merged panel

Even with zero rows, validate the schema and key structure. When rows are present, run full validation.

In [ ]:
print('=== MERGED PANEL VALIDATION ===')
print()

# 1. Column list
print('Columns:')
for c in merged.columns:
    print(f'  {c}: {merged[c].dtype}')
print()

# 2. Missingness
if len(merged) > 0:
    missing = merged.isnull().sum()
    missing_nonzero = missing[missing > 0]
    if len(missing_nonzero) == 0:
        print('Missingness: 0 missing values in any column. ✓')
    else:
        print('WARNING — Missing values found:')
        print(missing_nonzero)
    print()

    # 3. Key uniqueness: each (date_eu, days) pair should appear exactly once
    key_dupes = merged.duplicated(subset=['date_eu', 'days']).sum()
    if key_dupes == 0:
        print('Key uniqueness (date_eu, days): no duplicates. ✓')
    else:
        print(f'WARNING — {key_dupes} duplicate (date_eu, days) pairs found.')
    print()

    # 4. Summary statistics
    smile_cols = ['eu_atm_iv', 'eu_skew', 'eu_curvature',
                  'us_atm_iv', 'us_skew', 'us_curvature']
    available = [c for c in smile_cols if c in merged.columns]
    print('Summary statistics (smile parameters):')
    print(merged[available].describe().round(6))
else:
    print('Schema validation only (zero rows):')
    print(f'  Number of columns: {len(merged.columns)} ✓')
    print(f'  Expected columns present: ✓')
    print()
    print('Full validation will run automatically when rows are present.')

log['steps']['step6_validation'] = {
    'n_rows': len(merged),
    'n_cols': len(merged.columns),
    'columns': merged.columns.tolist(),
    'validation_note': 'full validation skipped (zero rows)' if len(merged) == 0 else 'full validation run',
}

---
## Step 7 — Save output and log

In [ ]:
# Output path
out_path = os.path.join(OUTPUT_DIR, f'panel_merged__{RUN_TIMESTAMP}.csv')
log_path = os.path.join(LOG_DIR, f'04_merge_log__{RUN_TIMESTAMP}.json')

# Save merged panel (even if zero rows — schema is preserved)
merged.to_csv(out_path, index=False)
print(f'Merged panel saved: {out_path}')
print(f'  Rows: {len(merged)}')
print(f'  Cols: {len(merged.columns)}')
print()

# Save log
log['output_file'] = out_path
log['status'] = 'complete'
log['zero_rows_note'] = (
    'Merged panel has zero rows because EU (2013) and US (2014) sample windows do not overlap. '
    'This is expected. The merge logic is correct. '
    'Re-run this notebook with overlapping data to produce a populated analysis-ready file.'
    if len(merged) == 0 else 'Merged panel populated successfully.'
)

with open(log_path, 'w') as f:
    json.dump(log, f, indent=2, default=str)
print(f'Log saved: {log_path}')

---
## Step 8 — Summary and next steps

**What this notebook built:**
- A validated merge pipeline that correctly aligns EU (day $t+1$) with US (day $t$) using the next-trading-day convention.
- Column prefixing (`eu_` / `us_`) to make the regression dataset unambiguous.
- An explicit trading calendar built from observed dates — no hardcoded calendar assumptions.
- A correctly structured output schema ready for notebook 05.

**Current limitation:**
- EU sample: 2013-03-01 to 2013-03-15 (Adidas options, IvyDB Europe sample)
- US sample: 2014-01-01 to 2014-04-30 (Apple options, IvyDB US sample)
- Zero overlapping dates → zero rows in merged panel.

**To get a populated merged panel, one of the following is needed:**
1. Full OptionMetrics access via WRDS (`optionm_all`, `optionm_europe_all`) — contact v.maccatrozzo@uva.nl
2. Refinitiv Eikon access (physical terminal, Roeterseiland campus library — book via llcrec@uva.nl)
3. PHLX currency options on WRDS (confirmed accessible — FX options, different asset class)

**Next:** Notebook 05 — Spillover Regression. Will load `panel_merged__<timestamp>.csv` and run the regression once rows are present.